# Loading data from BQ

In [2]:
from google.cloud import bigquery as bq
import pandas as pd

bq_client = bq.Client()

def load_bq_table_to_dataframe(table_path: str):
    """
    Loads data from a specified BigQuery table into a pandas DataFrame.

    Args:
        table_path (str): The full path to the BigQuery table (e.g., 'project.dataset.table').

    Returns:
        pandas.DataFrame: A DataFrame containing the data from the BigQuery table.
    """
    query_string = f"SELECT * FROM `{table_path}`"
    print(f"Executing BigQuery query for table: {table_path}")
    bq_query_job = bq_client.query(query_string, job_config=bq.QueryJobConfig())
    return bq_query_job.to_dataframe()

# Define the table paths (as strings)
base_path = 'ingka-ff-somdata-prod.OMDA_Analytics.'
path = f'{base_path}.auto_recovery_success_rate_analysis'

# Load data frames
df = load_bq_table_to_dataframe(path)

print("DataFrames loaded successfully!")

Executing BigQuery query for table: ingka-ff-somdata-prod.OMDA_Analytics..auto_recovery_success_rate_analysis


c:\Users\NILAV\OneDrive - IKEA\Documents\Project folder\Auto-recovery success rate analysis\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


DataFrames loaded successfully!


# Importing libraries

In [3]:
# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


# Analysis

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ===================================================================
# CONFIGURATION
# ===================================================================
OUTPUT_PATH = ""  # <-- FYLL I DIN SÖKVÄG HÄR, t.ex. "/path/to/output/" eller "C:/Users/yourname/output/"

# Skapa output-mapp om den inte finns
if OUTPUT_PATH and not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

# Säkerställ att event_time är datetime
df['event_time'] = pd.to_datetime(df['event_time'])

# Sortera data för att säkerställa korrekt ordning
df = df.sort_values(['country_code', 'order_no', 'service_prime_line_no', 'event_time'])

# Skapa unik identifierare för varje delivery
df['delivery_id'] = df['country_code'].astype(str) + '_' + df['order_no'].astype(str) + '_' + df['service_prime_line_no'].astype(str)

# ===================================================================
# DATA QUALITY CHECK
# ===================================================================

print("=" * 80)
print("DATA QUALITY VALIDATION")
print("=" * 80)

# Check: Är befintliga flaggor konsekventa?
quality_check = df.groupby('delivery_id').agg({
    'ever_successful': lambda x: x.nunique(),
    'never_successful': lambda x: x.nunique()
}).reset_index()

inconsistent_ever = (quality_check['ever_successful'] > 1).sum()
inconsistent_never = (quality_check['never_successful'] > 1).sum()

print(f"Deliveries with inconsistent 'ever_successful' flag: {inconsistent_ever:,}")
print(f"Deliveries with inconsistent 'never_successful' flag: {inconsistent_never:,}")

if inconsistent_ever > 0 or inconsistent_never > 0:
    print("⚠️  WARNING: Inconsistent flags detected. Recalculating from successful_AR events.")
else:
    print("✓ Flags are consistent across all attempts for each delivery")

print()

# ===================================================================
# ANALYS 1 & 2: Deliveries som är ever/never successfully auto-recovered
# ===================================================================

delivery_summary = df.groupby('delivery_id').agg({
    'successful_AR': 'sum',  # Totalt antal successful events per delivery
    'country_code': 'first'
}).reset_index()

# Beräkna korrekta flaggor baserat på faktiska successful events
delivery_summary['ever_successful'] = (delivery_summary['successful_AR'] > 0).astype(int)
delivery_summary['never_successful'] = (delivery_summary['successful_AR'] == 0).astype(int)

total_deliveries = len(delivery_summary)
ever_successful_count = delivery_summary['ever_successful'].sum()
never_successful_count = delivery_summary['never_successful'].sum()
ever_successful_pct = (ever_successful_count / total_deliveries) * 100
never_successful_pct = (never_successful_count / total_deliveries) * 100

# Validering: Summan ska vara exakt 100%
assert ever_successful_count + never_successful_count == total_deliveries, "Flaggorna summerar inte till totalt antal deliveries!"

print("=" * 80)
print("OVERALL AUTO-RECOVERY SUCCESS")
print("=" * 80)
print(f"Total unique deliveries: {total_deliveries:,}")
print(f"Ever successfully auto-recovered: {ever_successful_count:,} ({ever_successful_pct:.2f}%)")
print(f"Never successfully auto-recovered: {never_successful_count:,} ({never_successful_pct:.2f}%)")
print(f"✓ Validation: {ever_successful_pct + never_successful_pct:.2f}% (should be 100.00%)")
print()

# Tabell 1: Overall Success Summary
overall_summary = pd.DataFrame({
    'Metric': ['Total Deliveries', 'Ever Successfully Auto-Recovered', 'Never Successfully Auto-Recovered'],
    'Count': [total_deliveries, ever_successful_count, never_successful_count],
    'Percentage': [100.0, ever_successful_pct, never_successful_pct]
})

# ===================================================================
# ANALYS 3: Deliveries successfully auto-recovered at FIRST attempt
# ===================================================================

first_attempts = df[df['ar_attempt_seq'] == 1].copy()
first_attempt_successful = first_attempts['successful_AR'].sum()
first_attempt_total = len(first_attempts)
first_attempt_successful_pct = (first_attempt_successful / first_attempt_total) * 100 if first_attempt_total > 0 else 0

print("=" * 80)
print("FIRST ATTEMPT SUCCESS")
print("=" * 80)
print(f"Deliveries with first attempt: {first_attempt_total:,}")
print(f"Successfully auto-recovered at first attempt: {first_attempt_successful:,} ({first_attempt_successful_pct:.2f}%)")
print()

# Tabell 2: First Attempt Analysis
first_attempt_summary = pd.DataFrame({
    'Metric': ['Total First Attempts', 'Successful at First Attempt', 'Failed at First Attempt'],
    'Count': [first_attempt_total, first_attempt_successful, first_attempt_total - first_attempt_successful],
    'Percentage': [100.0, first_attempt_successful_pct, 100 - first_attempt_successful_pct]
})

# ===================================================================
# ANALYS 4: Deliveries successfully auto-recovered at n+1 attempts
# ===================================================================

successful_deliveries = df[df['successful_AR'] == 1].copy()
first_success = successful_deliveries.groupby('delivery_id').agg({
    'ar_attempt_seq': 'min'
}).reset_index()
first_success.columns = ['delivery_id', 'success_at_attempt']

attempt_distribution = first_success['success_at_attempt'].value_counts().sort_index()

print("=" * 80)
print("SUCCESS BY ATTEMPT NUMBER")
print("=" * 80)
total_ever_successful = len(first_success)
for attempt, count in attempt_distribution.items():
    pct = (count / total_ever_successful) * 100
    print(f"Attempt {attempt}: {count:,} deliveries ({pct:.2f}%)")
print()

later_attempts_count = len(first_success[first_success['success_at_attempt'] > 1])
later_attempts_pct = (later_attempts_count / total_ever_successful) * 100 if total_ever_successful > 0 else 0

print(f"Successfully auto-recovered at attempt 2 or later: {later_attempts_count:,} ({later_attempts_pct:.2f}%)")
print()

# Tabell 3: Success by Attempt Number
attempt_dist_df = pd.DataFrame({
    'Attempt_Number': attempt_distribution.index,
    'Count': attempt_distribution.values,
    'Percentage': (attempt_distribution.values / total_ever_successful * 100).round(2)
})

# ===================================================================
# ANALYS 5: Om första försöket misslyckas, hur troligt är det att lyckas senare?
# ===================================================================

first_attempt_failed = first_attempts[first_attempts['successful_AR'] == 0]['delivery_id'].unique()
first_failed_df = delivery_summary[delivery_summary['delivery_id'].isin(first_attempt_failed)]
first_failed_total = len(first_failed_df)
first_failed_later_success = first_failed_df['ever_successful'].sum()
recovery_after_first_fail_pct = (first_failed_later_success / first_failed_total) * 100 if first_failed_total > 0 else 0

print("=" * 80)
print("RECOVERY AFTER FIRST ATTEMPT FAILURE")
print("=" * 80)
print(f"Deliveries that failed at first attempt: {first_failed_total:,}")
print(f"Later successfully auto-recovered: {first_failed_later_success:,} ({recovery_after_first_fail_pct:.2f}%)")
print(f"Never recovered after first failure: {first_failed_total - first_failed_later_success:,} ({100 - recovery_after_first_fail_pct:.2f}%)")
print()

# Tabell 4: Recovery After First Failure
recovery_after_failure = pd.DataFrame({
    'Metric': ['Failed at First Attempt', 'Later Successfully Recovered', 'Never Recovered'],
    'Count': [first_failed_total, first_failed_later_success, first_failed_total - first_failed_later_success],
    'Percentage': [100.0, recovery_after_first_fail_pct, 100 - recovery_after_first_fail_pct]
})

# ===================================================================
# BONUS: Detaljerad sammanfattning per försöksnummer
# ===================================================================

print("=" * 80)
print("DETAILED BREAKDOWN BY ATTEMPT SEQUENCE")
print("=" * 80)

attempt_analysis = df.groupby('ar_attempt_seq').agg({
    'delivery_id': 'nunique',
    'successful_AR': 'sum',
    'unsuccessful_AR': 'sum'
}).reset_index()

attempt_analysis.columns = ['Attempt', 'Unique_Deliveries', 'Successful', 'Unsuccessful']
attempt_analysis['Success_Rate_%'] = (attempt_analysis['Successful'] / attempt_analysis['Unique_Deliveries'] * 100).round(2)

print(attempt_analysis.to_string(index=False))
print()

# ===================================================================
# EXTRA: Analys per land
# ===================================================================

country_analysis = delivery_summary.groupby('country_code').agg({
    'delivery_id': 'count',
    'ever_successful': 'sum',
    'never_successful': 'sum'
}).reset_index()

country_analysis.columns = ['Country_Code', 'Total_Deliveries', 'Ever_Successful', 'Never_Successful']
country_analysis['Success_Rate_%'] = (country_analysis['Ever_Successful'] / country_analysis['Total_Deliveries'] * 100).round(2)
country_analysis = country_analysis.sort_values('Total_Deliveries', ascending=False)

print("=" * 80)
print("ANALYSIS BY COUNTRY")
print("=" * 80)
print(country_analysis.to_string(index=False))
print()

# ===================================================================
# EXPORT TO EXCEL
# ===================================================================

excel_path = os.path.join(OUTPUT_PATH, 'auto_recovery_analysis.xlsx')

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    overall_summary.to_excel(writer, sheet_name='Overall Summary', index=False)
    first_attempt_summary.to_excel(writer, sheet_name='First Attempt', index=False)
    attempt_dist_df.to_excel(writer, sheet_name='Success by Attempt', index=False)
    recovery_after_failure.to_excel(writer, sheet_name='Recovery After Failure', index=False)
    attempt_analysis.to_excel(writer, sheet_name='Detailed by Attempt', index=False)
    country_analysis.to_excel(writer, sheet_name='Country Analysis', index=False)

print(f"✓ Excel file saved to: {excel_path}")
print()

# ===================================================================
# VISUALIZATIONS
# ===================================================================

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
colors_ikea = ['#0058A3', '#FFDB00', '#111111', '#E0E0E0', '#8B8B8B']

# ---- VISUALIZATION 1: Overall Success/Failure ----
fig1, ax1 = plt.subplots(figsize=(10, 6))
categories = ['Ever Successful', 'Never Successful']
values = [ever_successful_count, never_successful_count]
colors = [colors_ikea[0], colors_ikea[2]]

bars = ax1.bar(categories, values, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_ylabel('Number of Deliveries', fontsize=12, fontweight='bold')
ax1.set_title('Overall Auto-Recovery Success Rate', fontsize=14, fontweight='bold', pad=20)
ax1.set_ylim(0, max(values) * 1.15)

# Add value labels on bars
for bar, val, pct in zip(bars, values, [ever_successful_pct, never_successful_pct]):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{val:,}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
fig1_path = os.path.join(OUTPUT_PATH, 'overall_success_rate.png')
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
print(f"✓ Visualization saved: {fig1_path}")
plt.close()

# ---- VISUALIZATION 2: First Attempt Success ----
fig2, ax2 = plt.subplots(figsize=(10, 6))
categories2 = ['Successful', 'Failed']
values2 = [first_attempt_successful, first_attempt_total - first_attempt_successful]
colors2 = [colors_ikea[1], colors_ikea[4]]

bars2 = ax2.bar(categories2, values2, color=colors2, edgecolor='black', linewidth=1.2)
ax2.set_ylabel('Number of Deliveries', fontsize=12, fontweight='bold')
ax2.set_title('First Attempt Auto-Recovery Success Rate', fontsize=14, fontweight='bold', pad=20)
ax2.set_ylim(0, max(values2) * 1.15)

for bar, val in zip(bars2, values2):
    height = bar.get_height()
    pct = (val / first_attempt_total * 100)
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{val:,}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
fig2_path = os.path.join(OUTPUT_PATH, 'first_attempt_success.png')
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
print(f"✓ Visualization saved: {fig2_path}")
plt.close()

# ---- VISUALIZATION 3: Success by Attempt Number ----
fig3, ax3 = plt.subplots(figsize=(12, 6))
attempts = attempt_dist_df['Attempt_Number'].values
counts = attempt_dist_df['Count'].values

bars3 = ax3.bar(attempts, counts, color=colors_ikea[0], edgecolor='black', linewidth=1.2, alpha=0.8)
ax3.set_xlabel('Attempt Number', fontsize=12, fontweight='bold')
ax3.set_ylabel('Number of Deliveries', fontsize=12, fontweight='bold')
ax3.set_title('Distribution of Success by Attempt Number', fontsize=14, fontweight='bold', pad=20)
ax3.set_xticks(attempts)
ax3.set_ylim(0, max(counts) * 1.15)

for bar, val, pct in zip(bars3, counts, attempt_dist_df['Percentage'].values):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
             f'{val:,}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
fig3_path = os.path.join(OUTPUT_PATH, 'success_by_attempt.png')
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
print(f"✓ Visualization saved: {fig3_path}")
plt.close()

# ---- VISUALIZATION 4: Recovery After First Failure ----
fig4, ax4 = plt.subplots(figsize=(10, 6))
labels4 = ['Later Recovered', 'Never Recovered']
sizes4 = [first_failed_later_success, first_failed_total - first_failed_later_success]
colors4 = [colors_ikea[1], colors_ikea[2]]
explode4 = (0.05, 0)

wedges, texts, autotexts = ax4.pie(sizes4, explode=explode4, labels=labels4, colors=colors4,
                                     autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'},
                                     wedgeprops={'edgecolor': 'black', 'linewidth': 1.5})

ax4.set_title('Recovery Rate After First Attempt Failure', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
fig4_path = os.path.join(OUTPUT_PATH, 'recovery_after_first_failure.png')
plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
print(f"✓ Visualization saved: {fig4_path}")
plt.close()

# ---- VISUALIZATION 5: Success Rate by Attempt (Line Chart) ----
fig5, ax5 = plt.subplots(figsize=(12, 6))

attempt_nums = attempt_analysis['Attempt'].values
success_rates = attempt_analysis['Success_Rate_%'].values

ax5.plot(attempt_nums, success_rates, marker='o', linewidth=2.5, markersize=8, 
         color=colors_ikea[0], markerfacecolor=colors_ikea[1], markeredgecolor='black', 
         markeredgewidth=1.5)

ax5.set_xlabel('Attempt Number', fontsize=12, fontweight='bold')
ax5.set_ylabel('Success Rate (%)', fontsize=12, fontweight='bold')
ax5.set_title('Success Rate Trend by Attempt Number', fontsize=14, fontweight='bold', pad=20)
ax5.grid(True, alpha=0.3, linestyle='--')
ax5.set_ylim(0, 100)
ax5.set_xticks(attempt_nums)

# Add value labels
for x, y in zip(attempt_nums, success_rates):
    ax5.text(x, y + 2, f'{y:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
fig5_path = os.path.join(OUTPUT_PATH, 'success_rate_trend.png')
plt.savefig(fig5_path, dpi=300, bbox_inches='tight')
print(f"✓ Visualization saved: {fig5_path}")
plt.close()

# ---- VISUALIZATION 6: Top Countries by Volume ----
top_countries = country_analysis.head(10)

fig6, ax6 = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(top_countries))
deliveries = top_countries['Total_Deliveries'].values
countries = top_countries['Country_Code'].values

bars6 = ax6.barh(y_pos, deliveries, color=colors_ikea[0], edgecolor='black', linewidth=1.2)
ax6.set_yticks(y_pos)
ax6.set_yticklabels(countries, fontsize=11)
ax6.invert_yaxis()
ax6.set_xlabel('Number of Deliveries', fontsize=12, fontweight='bold')
ax6.set_title('Top 10 Countries by Delivery Volume', fontsize=14, fontweight='bold', pad=20)

# Add value labels
for i, (bar, val, success_rate) in enumerate(zip(bars6, deliveries, top_countries['Success_Rate_%'].values)):
    width = bar.get_width()
    ax6.text(width, bar.get_y() + bar.get_height()/2.,
             f' {val:,} (SR: {success_rate:.1f}%)',
             ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
fig6_path = os.path.join(OUTPUT_PATH, 'top_countries_volume.png')
plt.savefig(fig6_path, dpi=300, bbox_inches='tight')
print(f"✓ Visualization saved: {fig6_path}")
plt.close()

# ---- VISUALIZATION 7: Country Success Rates (for countries with >100 deliveries) ----
significant_countries = country_analysis[country_analysis['Total_Deliveries'] >= 100].sort_values('Success_Rate_%', ascending=True)

if len(significant_countries) > 0:
    fig7, ax7 = plt.subplots(figsize=(12, max(6, len(significant_countries) * 0.4)))
    
    y_pos7 = np.arange(len(significant_countries))
    success_rates7 = significant_countries['Success_Rate_%'].values
    countries7 = significant_countries['Country_Code'].values
    
    # Color bars based on success rate
    colors7 = [colors_ikea[0] if sr >= 50 else colors_ikea[4] for sr in success_rates7]
    
    bars7 = ax7.barh(y_pos7, success_rates7, color=colors7, edgecolor='black', linewidth=1.2)
    ax7.set_yticks(y_pos7)
    ax7.set_yticklabels(countries7, fontsize=10)
    ax7.invert_yaxis()
    ax7.set_xlabel('Success Rate (%)', fontsize=12, fontweight='bold')
    ax7.set_title('Success Rate by Country (Min. 100 Deliveries)', fontsize=14, fontweight='bold', pad=20)
    ax7.set_xlim(0, 100)
    ax7.axvline(x=50, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='50% Threshold')
    ax7.legend(loc='lower right')
    
    # Add value labels
    for bar, val, total in zip(bars7, success_rates7, significant_countries['Total_Deliveries'].values):
        width = bar.get_width()
        ax7.text(width + 1, bar.get_y() + bar.get_height()/2.,
                 f'{val:.1f}% (n={total:,})',
                 ha='left', va='center', fontsize=8, fontweight='bold')
    
    plt.tight_layout()
    fig7_path = os.path.join(OUTPUT_PATH, 'country_success_rates.png')
    plt.savefig(fig7_path, dpi=300, bbox_inches='tight')
    print(f"✓ Visualization saved: {fig7_path}")
    plt.close()

# ===================================================================
# SUMMARY DASHBOARD DATA
# ===================================================================

summary_stats = pd.DataFrame({
    'Key Metric': [
        'Total Deliveries',
        'Ever Successfully Auto-Recovered (%)',
        'Never Successfully Auto-Recovered (%)',
        'Success at First Attempt (%)',
        'Success at Later Attempts (%)',
        'Recovery Rate After First Failure (%)',
        'Average Attempts to Success'
    ],
    'Value': [
        f"{total_deliveries:,}",
        f"{ever_successful_pct:.2f}%",
        f"{never_successful_pct:.2f}%",
        f"{first_attempt_successful_pct:.2f}%",
        f"{later_attempts_pct:.2f}%",
        f"{recovery_after_first_fail_pct:.2f}%",
        f"{first_success['success_at_attempt'].mean():.2f}"
    ]
})

print("=" * 80)
print("KEY METRICS SUMMARY")
print("=" * 80)
print(summary_stats.to_string(index=False))
print()

# Add summary to Excel
with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a') as writer:
    summary_stats.to_excel(writer, sheet_name='Key Metrics', index=False)

print(f"✓ Summary metrics added to Excel file")
print()

# ===================================================================
# FINAL MESSAGE
# ===================================================================

print("=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print(f"All outputs saved to: {OUTPUT_PATH}")
print()
print("Generated files:")
print(f"  📊 Excel: auto_recovery_analysis.xlsx")
print(f"  📈 Visualizations:")
print(f"     - overall_success_rate.png")
print(f"     - first_attempt_success.png")
print(f"     - success_by_attempt.png")
print(f"     - recovery_after_first_failure.png")
print(f"     - success_rate_trend.png")
print(f"     - top_countries_volume.png")
if len(significant_countries) > 0:
    print(f"     - country_success_rates.png")
print()
print("=" * 80)


DATA QUALITY VALIDATION
Deliveries with inconsistent 'ever_successful' flag: 0
Deliveries with inconsistent 'never_successful' flag: 0
✓ Flags are consistent across all attempts for each delivery

OVERALL AUTO-RECOVERY SUCCESS
Total unique deliveries: 38,066
Ever successfully auto-recovered: 19,568 (51.41%)
Never successfully auto-recovered: 18,498 (48.59%)
✓ Validation: 100.00% (should be 100.00%)

FIRST ATTEMPT SUCCESS
Deliveries with first attempt: 38,066
Successfully auto-recovered at first attempt: 17,992 (47.27%)

SUCCESS BY ATTEMPT NUMBER
Attempt 1: 17,992 deliveries (91.95%)
Attempt 2: 1,333 deliveries (6.81%)
Attempt 3: 167 deliveries (0.85%)
Attempt 4: 61 deliveries (0.31%)
Attempt 5: 11 deliveries (0.06%)
Attempt 6: 1 deliveries (0.01%)
Attempt 8: 1 deliveries (0.01%)
Attempt 9: 1 deliveries (0.01%)
Attempt 18: 1 deliveries (0.01%)

Successfully auto-recovered at attempt 2 or later: 1,576 (8.05%)

RECOVERY AFTER FIRST ATTEMPT FAILURE
Deliveries that failed at first attempt: 